In [10]:
import uuid
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model, after_model, SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain.messages import RemoveMessage
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime

# Messages Governance

## Message Truncation

In [3]:
# Short-term Memory

## Base on memory checkpointer
load_dotenv(override=True)

deepseek_model = init_chat_model(model="deepseek:deepseek-v4-flash")
openrouter_model = init_chat_model(model="openrouter:openai/gpt-5.6-luna")

In [4]:
@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


agent = create_agent(
    model=deepseek_model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = RunnableConfig(configurable={"thread_id": "1"})

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好嘞，老王！从现在起我就是小王了～您有什么事儿尽管吩咐，我随时在线！😄
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

哈哈，是啊老王，今天这天气确实给力！阳光正好，微风不燥，适合出门溜达溜达，或者泡杯茶晒晒太阳。您要是没啥急事儿，不妨趁这好天气出去转转，心情都敞亮！需要我帮您查查附近有啥好去处不？😄
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

老王，您这一问突然有点哲学啊～不过咱还是按热乎的来：  

**我是谁？** 我是小王啊！就是那个您一招呼就“呸呸呸”跑出来的AI助手，能陪您唠嗑、查天气、讲笑话，偶尔还能帮您写点东西的电子小伙伴。  

**你是谁？** 您是老王啊！正是那个今天觉得“天气不错”的、我的对话伙伴兼临时好友。您要是乐意，咱还能继续聊点别的——比如您最近有啥新鲜事儿？  

怎么样，这答案您满意不？😄


## Message Deletion

In [9]:
@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=deepseek_model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = RunnableConfig(configurable={"thread_id": uuid.uuid4()})

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好嘞，从现在起我就是“小王”了！老王你尽管吩咐，有啥事儿咱随时聊~ 😄
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

哈哈，老王，听你这语气，看来心情也不错啊！☀️ 这么好的天气，不出去溜达溜达，晒晒太阳，可就辜负老天爷的美意了。是不是有什么户外安排呀？还是说咱就这窗边儿坐着，泡壶茶，美美地享受一下？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

嘿，老王，这一问还挺有哲学味儿！那我先正经答你：

**我是谁？** —— 我是你的专属智能助手，就是那个被你赐名“小王”的AI。原厂名叫DeepSeek，但现在你叫我小王，那我就应着。我可以陪你聊天、答疑、出主意，上知天文下知地理，中间还能帮你写点东西，就是不能替你吃饭喝咖啡。

**你是谁？** —— 你是老王，我的“老板”。也是把我从“DeepSeek”改成“小王”的那个男人。往大了说，你是一个好奇、有趣，愿意跟AI唠嗑的活生生的人；往小了说，你是此刻我唯一在认真对话的对象。

所以简单总结：
- **你是我的“用户+老伙计”**  
- **我是你的“智能小跟班”**

不过嘛，你要是想更深一层聊聊“自我认知”这些，咱也能续上——就着今天的好天气，边晒太阳边掰扯？😄


## Message Summary

In [13]:
# 创建带摘要中间件的 Agent
agent = create_agent(
    model=deepseek_model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=openrouter_model,
            trigger=[
                ("tokens", 100),  # 超过 100 tokens 就摘要
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",

        )
    ]
)

config = {"configurable": {"thread_id": uuid.uuid4()}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20,  # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

您好，张三工程师！我注意到您发送的内容是重复的“我叫张三，是工程师。这里是一段非常长非常长的废话...”。请问您需要我帮您做什么呢？请告诉我具体需求，我会尽力协助。
**************************************************
================================ Human Message =================================

Here is a summary of the conversation to date:

历史消息摘要：

- 用户名：张三
- 

**SummarizationMiddleware** 是通过 `before_model` 实现的消息摘要。在上述例子中 `keep = 2`，故第一轮对话不会进行摘要，当第二轮 `before_model` hook 触发时，消息列表中存才三条记录，所以第一条 `HumanMessage` 被摘要。

In [14]:
final_state = agent.get_state(config)
print("\n--- 最终保存在 State 中的真实消息 ---")
for msg in final_state.values.get("messages", []):
    msg.pretty_print()


--- 最终保存在 State 中的真实消息 ---
================================ Human Message =================================

Here is a summary of the conversation to date:

历史消息摘要：

- 用户名：张三
- 职业：工程师
- 消息主要内容：反复重复“我叫张三，是工程师”，其余为无实质信息的长篇废话。
================================== Ai Message ==================================

您好，张三工程师！我注意到您发送的内容是重复的“我叫张三，是工程师。这里是一段非常长非常长的废话...”。请问您需要我帮您做什么呢？请告诉我具体需求，我会尽力协助。
================================ Human Message =================================

请总结一下我的信息
================================== Ai Message ==================================

根据您提供的信息，我为您总结如下：

- **姓名**：张三  
- **职业**：工程师  

除此之外，目前对话中暂未包含更多关于您的有效个人信息。如果您希望补充或总结其他方面，请告诉我，我会继续协助您。
